# Pandas — short tutorial

This notebook introduces the **core ideas** behind [pandas](https://pandas.pydata.org/): one-dimensional **Series**, two-dimensional **DataFrames**, **indexing**, **missing values**, and **group-by** operations. Run each cell in order.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 10)
print(pd.__version__)

2.2.3


## 1. Series: labeled 1D arrays

A **Series** is a sequence of values with an **index** (labels). Think “column with row names.”

In [2]:
s = pd.Series([10, 20, 30], index=["a", "b", "c"], name="count")
print(s)
print("values:", s.values)
print("index:", s.index.tolist())
print("element b:", s.loc["b"])          # label-based
print("first row:", s.iloc[0])           # position-based

a    10
b    20
c    30
Name: count, dtype: int64
values: [10 20 30]
index: ['a', 'b', 'c']
element b: 20
first row: 10


## 2. DataFrame: table of columns

A **DataFrame** is a dict-like collection of Series that share the same index — rows × columns.

In [3]:
df = pd.DataFrame({
    "name": ["Ada", "Bob", "Cara"],
    "age": [28, 34, 22],
    "score": [91.5, 88.0, np.nan],
})
df

,name,age,score
0,Ada,28,91.5
1,Bob,34,88.0
2,Cara,22,NaN


### Shape, dtypes, and quick profile

`info()` and `describe()` summarize structure and numeric columns.

In [4]:
print(df.shape, df.dtypes.to_dict())
display(df.head(2))
display(df.describe(include="all"))

(3, 3) {'name': dtype('O'), 'age': dtype('int64'), 'score': dtype('float64')}


,name,age,score
0,Ada,28,91.5
1,Bob,34,88.0


,name,age,score
count,3,3.0,2.000000
unique,3,NaN,NaN
top,Ada,NaN,NaN
freq,1,NaN,NaN
mean,NaN,28.0,89.750000
std,NaN,6.0,2.474874
min,NaN,22.0,88.000000
25%,NaN,25.0,88.875000
50%,NaN,28.0,89.750000
75%,NaN,31.0,90.625000


## 3. Selecting columns and rows

- **`df["col"]`** — one column (Series).
- **`df[["a","b"]]`** — several columns (DataFrame).
- **`loc`** — select by **labels** (slice rows/cols by index names).
- **`iloc`** — select by **integer position** (0-based).
- **Boolean mask** — `df[df["age"] > 25]` filters rows.

In [5]:
# columns
display(df["name"])
display(df[["name", "age"]])

# filter rows
display(df[df["age"] >= 30])

# loc / iloc (set index for nicer loc)
df_idx = df.set_index("name")
display(df_idx.loc["Bob":"Cara", ["age", "score"]])  # label slice, inclusive
display(df_idx.iloc[0:2, 0:2])                           # positions: rows 0–1, cols 0–1

0     Ada
1     Bob
2    Cara
Name: name, dtype: object

,name,age
0,Ada,28
1,Bob,34
2,Cara,22


,name,age,score
1,Bob,34,88.0


,age,score
name,,
Bob,34,88.0
Cara,22,NaN


,age,score
name,,
Ada,28,91.5
Bob,34,88.0


## 4. Adding and deriving columns

Assign like a dict key; vectorized ops use underlying NumPy when dtypes allow.

In [6]:
df2 = df.copy()
df2["passed"] = df2["score"] >= 90
df2["age_group"] = pd.cut(df2["age"], bins=[0, 30, 100], labels=["≤30", ">30"])
df2

,name,age,score,passed,age_group
0,Ada,28,91.5,True,≤30
1,Bob,34,88.0,False,>30
2,Cara,22,NaN,False,≤30


## 5. Missing data (`NaN`)

**`isna` / `notna`** detect missing values. **`fillna`** imputes; **`dropna`** removes rows (or columns) with NA.

In [7]:
missing = df.copy()
display(missing.isna())
display(missing.fillna({"score": missing["score"].mean()}))
display(missing.dropna(subset=["score"]))

,name,age,score
0,False,False,False
1,False,False,False
2,False,False,True


,name,age,score
0,Ada,28,91.50
1,Bob,34,88.00
2,Cara,22,89.75


,name,age,score
0,Ada,28,91.5
1,Bob,34,88.0


## 6. Sorting and ranking

In [8]:
df.sort_values("age", ascending=False)
# df.sort_index() for row order by index

,name,age,score
1,Bob,34,88.0
0,Ada,28,91.5
2,Cara,22,NaN


## 7. Group-by: split → apply → combine

Group rows by one or more keys, then aggregate each group (mean, sum, size, custom functions).

In [9]:
sales = pd.DataFrame({
    "region": ["N", "N", "S", "S", "N"],
    "product": ["A", "B", "A", "B", "A"],
    "amount": [100, 150, 200, 50, 120],
})
g = sales.groupby("region", observed=True)
display(g["amount"].sum())
display(sales.groupby(["region", "product"], observed=True)["amount"].mean().unstack())

region
N    370
S    250
Name: amount, dtype: int64

product,A,B
region,,
N,110.0,150.0
S,200.0,50.0


## 8. Combining tables: `concat` and `merge`

- **`pd.concat`** — stack DataFrames vertically (`axis=0`) or side-by-side (`axis=1`).
- **`pd.merge`** — SQL-style **joins** on key columns (`how`: `left`, `right`, `inner`, `outer`).

In [10]:
left = pd.DataFrame({"id": [1, 2, 3], "val": ["x", "y", "z"]})
right = pd.DataFrame({"id": [2, 3, 4], "meta": [10, 20, 30]})
display(pd.merge(left, right, on="id", how="inner"))

,id,val,meta
0,2,y,10
1,3,z,20


## 9. Reading and writing files (typical workflow)

For real data you often use **`read_csv`**, **`read_excel`**, **`read_parquet`**, etc., and **`to_csv`** / **`to_parquet`** to save.

In [11]:
# Example (uncomment when you have a file path):
# df_io = pd.read_csv("data.csv")
# df_io.to_csv("out.csv", index=False)
print("Use pd.read_csv('path.csv') and df.to_csv('out.csv', index=False) in your projects.")

Use pd.read_csv('path.csv') and df.to_csv('out.csv', index=False) in your projects.


## Cheat sheet — what to remember

| Concept | Role |
|---------|------|
| **Series** | 1D labeled data |
| **DataFrame** | 2D table; columns are Series |
| **`loc` / `iloc`** | Label vs position indexing |
| **Boolean indexing** | `df[condition]` |
| **`isna` / `fillna` / `dropna`** | Missing values |
| **`groupby`** | Aggregations per group |
| **`merge` / `concat`** | Join and stack tables |

Next steps: time series (`pd.date_range`, resampling), `pivot_table`, `apply` / `map`, and the official [10 minutes to pandas](https://pandas.pydata.org/docs/user_guide/10min.html).